# XDA-IDS on Google Colab

Before you start: **Runtime → Change runtime type → T4 GPU**.

Run the cells top to bottom. Data comes straight from Kaggle, so no dataset upload is needed. Section 4 holds the long experiments; run **one of them per session** (free Colab disconnects after about 12 h, or about 90 min idle). Every result is saved to Drive as soon as each seed finishes.

## 1. GPU check, project code (GitHub) and results folder (Drive)

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU"

In [ ]:
# Get the project code from GitHub.
REPO = "https://github.com/<your-username>/xda-ids.git"   # the repo you published from VS Code
GITHUB_TOKEN = ""   # only needed if the repo is private (GitHub > Settings > Developer settings > Tokens)

import os
url = REPO.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO
if os.path.exists("/content/xda_ids"):
    !git -C /content/xda_ids pull -q
else:
    !git clone -q {url} /content/xda_ids
%cd /content/xda_ids

# Results go to Google Drive when it can be mounted, so they survive disconnects.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS = "/content/drive/MyDrive/xda_results"
except Exception as e:
    print("Drive not available, results stay on this machine (download them at the end):", e)
    RESULTS = "/content/xda_results"
os.makedirs(RESULTS, exist_ok=True)
!rm -rf results && ln -s {RESULTS} results
print("results ->", RESULTS)


## 2. Libraries
Everything except `kagglehub` and `river` is preinstalled on Colab.

In [ ]:
!pip install -q kagglehub river scikit-posthocs
import torch, shap, sklearn, river
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "| shap", shap.__version__)

## 3. Datasets (fetched from Kaggle, nothing to upload)

`configs/default.yaml` points at these Kaggle datasets; kagglehub downloads them onto the Colab machine:
- UNSW-NB15: `mrwellsdavid/unsw-nb15` (training-set + testing-set CSVs)
- WUSTL-IIoT-2021: `annaamalaiu/wustl-iiot-2021-dataset` (fallback `mostafa2372/wustl-iiot-2021`)

If Kaggle asks for a login, run the next cell once (Kaggle > Settings > API > Create New Token).

In [ ]:
# Only needed if a download fails with an authentication error.
# import kagglehub; kagglehub.login()

In [ ]:
# Fetch both datasets now (cached for the session) and show where they landed.
import sys; sys.path.insert(0, "src")
import yaml
from xda.data import resolve
CFG = "configs/default.yaml"
data_cfg = yaml.safe_load(open(CFG))["data"]
for key in ["unsw_train", "unsw_test", "wustl"]:
    print(key, "->", resolve(data_cfg[key]))

## 4. Checks (a few minutes)

In [ ]:
# Pipeline test on synthetic data, no datasets needed.
!python scripts/smoke_test.py

In [ ]:
# Class counts, duplicates and the 'id' / 'Target' leakage checks on the real data.
!python scripts/check_data.py --config {CFG}

In [ ]:
# Quick first look: 1 seed, 10 epochs, all five variants on UNSW multiclass.
!python scripts/run_static.py --config {CFG} --dataset unsw --task multiclass --seeds 1 --max-epochs 10 --out results/quick

## 5. Full experiments (one cell per session)

Each cell saves results per seed/scenario as it goes; finished runs are skipped when a cell is re-run.
Order follows the methodology: detector (Module A) → explanations (Module B) → drift monitor (Module C) → baselines → statistics.

### 5.1 Detector: XDA-IDS vs base paper, with ablation (E1, E2, Step 10)

In [ ]:
!python scripts/run_static.py --config {CFG} --dataset unsw --task multiclass --out results/static
!python scripts/run_static.py --config {CFG} --dataset unsw --task binary --out results/static

In [ ]:
!python scripts/run_static.py --config {CFG} --dataset wustl --task multiclass --out results/static
!python scripts/run_static.py --config {CFG} --dataset wustl --task binary --out results/static

### 5.2 Detector baselines: LSTM, Random Forest, XGBoost, RF + Isolation Forest hybrid (Step 6)

In [ ]:
for ds in ["unsw", "wustl"]:
    !python scripts/run_static.py --config {CFG} --dataset {ds} --task multiclass --variants lstm rf xgb hybrid_rf_if --out results/static

### 5.3 Explanation quality and SHAP-guided Top-k sweep (Step 4, Step 8, Step 11)

In [ ]:
for ds in ["unsw", "wustl"]:
    !python scripts/run_explain.py --config {CFG} --dataset {ds} --task multiclass --seed 1 --out results/explain

### 5.4 Drift monitor: unseen attack families (E3)

In [ ]:
for seed in [1, 2, 3, 4, 5]:
    !python scripts/run_drift.py --config {CFG} --family Exploits DoS Reconnaissance Fuzzers Backdoor --seed {seed} --out results/drift

### 5.5 Drift monitor: injected drift, four types + class mixture (E4)

In [ ]:
for seed in [1, 2, 3, 4, 5]:
    !python scripts/run_drift.py --config {CFG} --dataset unsw --drift sudden gradual incremental recurring --class-mix DoS --seed {seed} --out results/drift

In [ ]:
# WUSTL: injected drift plus its natural time-ordered stream (false alarms on real traffic)
for seed in [1, 2, 3]:
    !python scripts/run_drift.py --config {CFG} --dataset wustl --drift sudden gradual incremental recurring --chrono --seed {seed} --out results/drift

### 5.6 Retraining loop (Step 5.5) and window / sample-size sensitivity (Step 10)

In [ ]:
!python scripts/run_drift.py --config {CFG} --family Exploits DoS Backdoor --adapt --seed 1 --out results/drift_adapt

In [ ]:
for w, n in [(1000, 300), (10000, 300), (5000, 100), (5000, 1000)]:
    !python scripts/run_drift.py --config {CFG} --drift sudden gradual --window {w} --sample {n} --seed 1 --out results/drift_sensitivity

### 5.7 Cross-dataset transfer (E5)

Needs a few CICIoT2023 CSV parts (https://www.unb.ca/cic/datasets/iotdataset-2023.html, or a Kaggle mirror).
Put them on Drive and set the glob below.

In [ ]:
CICIOT = "/content/drive/MyDrive/ciciot2023/*.csv"
!python scripts/run_transfer.py --config {CFG} --ciciot "{CICIOT}" --seed 1 --out results/transfer

## 6. Statistics (Step 9) and failure analysis (Step 11)

In [ ]:
!python scripts/stats.py --static results/static --drift results/drift

In [ ]:
!python scripts/analyze_errors.py --static results/static --drift results/drift

## 7. Download results (only if Drive was not available)

In [ ]:
!cd /content && zip -qr xda_results.zip xda_results && ls -lh /content/xda_results.zip
try:
    from google.colab import files
    files.download("/content/xda_results.zip")
except Exception as e:
    print("Automatic download not available here; fetch /content/xda_results.zip from the Colab file browser.", e)
